# Machine Learning using Sentinel 2 images through OpenEO and Forest Inventory data

In this document the Sentinel 2 images that cover the burned area after a forest fire are downloaded and a Machine Learning algorithm is trained to classify the vegetation species before the fire.

1. Connect to openEO and define the Cube
2. Define ML pipeline
3. Train ML algorithm

In [88]:
import matplotlib.pyplot as plt
from PIL import Image
import openeo
from openeo.api.process import Parameter
# OpenEO UDP parameter management system
from openeo_udp import ParameterManager
import geopandas
import json
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

## Setup connection to OpenEO

In [ ]:
connection = openeo.connect(
    url="https://openeo.dataspace.copernicus.eu/"
).authenticate_oidc()

time_frame = ["2024-01-01,2024-12-30"]
time_post= ["2024-09-30", "2024-10-10"] # + last
time_pre=  ["2024-08-20", "2024-09-01"]  # + last

current_params = {
        "location_name": "Reriz e Gafanhao, North Portugal",
        "bounding_box": Parameter(
            "bounding_box",
            description="Fire area in 2024",
            default= {"west": -8.23, "south": 40.76, "east": -7.78, "north": 41}
),
        "time": Parameter(
            "time",
            description="Temporal range for data acquisition",
            # default=["2024-09-30", "2024-10-30"], # + last
            default=["2024-08-01", "2024-09-01"],  # + last
        ),
        "bands": Parameter(
            "bands",
            description="Sentinel-2 bands required for APA calculation",
            default=  ["B02","B03","B04","B05","B07","B08","B8A","B11","B12"]
#,
        ),
        "collection": Parameter(
            "collection",
            description="Data collection identifier",
            default="SENTINEL2_L2A",
        ),
        "cloud_cover": Parameter(
            "cloud_cover",
            description="Maximum cloud cover percentage",
            default=30,
        ),
    }


In [ ]:
def load_and_sample(params, time, res=100):
    """
    Input: Set of parameters, time frame, desired resolution
    Output: Cube that contains image
    """ 
    s2cube = connection.load_collection(
    params["collection"].default,
    temporal_extent=time,
    spatial_extent=params["bounding_box"].default,
    bands=params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= params["cloud_cover"].default,
    },
    )
    # s2cube = s2cube.reduce_dimension(dimension="t", reducer="last")
    s2cube = s2cube.resample_spatial(
        resolution=[res, res],
      method="near",
    )

    return s2cube



# def calculate_ndvi(data):
#      """ 
#      Input: Cube with bands 04 and 8
#      Output: NBR
#      """
#      B04, B08 = (
#           data[2],
#           data[5]
#           )
#      ndvi = (B08 - B04) / (B08 + B04)
#      return ndvi

## Define ML pipeline that takes as input the label path and the data cube

In [96]:
def load_labels(path):
    """Load the labels from the geopackage and return them as a GeoDataFrame."""
    # Load the labels from the geopackage
    labels = geopandas.read_file(path, layer = "nfi")
    return labels

def process_labels(labels):
    """Process the labels and return them as a GeoJSON object."""
    #Convert to Coordinate System of S2 in Portugal
    labels = labels.to_crs("EPSG:4326")  
    
    # drop "Other Forest" and "Non forest" labels
    labels = labels[labels["label"]!="Other Forest"]
    labels = labels[labels["label"]!="Non forest"]

    # Ensure right geometry type of labels
    labels["geometry"] = labels["geometry"].apply(lambda g: g.geoms[0] if g.geom_type == "MultiPoint" else g)
    
    # Convert to Geojson
    labels_gj = labels[["label", "geometry"]]
    geojson = json.loads(labels_gj.to_json())

    return labels, geojson


def load_features(s2cube, geojson):
    """Load the features from the satellite image and return them as a DataFrame. (Convert to vector data)"""
    # Define the features to be extracted
    features = s2cube.aggregate_spatial(geojson, reducer="mean")
    return features

def downlaod_features(features):
    """Download the features as a JSON file."""
    job = features.create_job(out_format="JSON")
    job.start_and_wait()
    job.get_results().download_files("features_output")
    with open("features_output/timeseries.json") as f:
        raw = json.load(f)
    return raw

def label_feature_matching(raw, nfi_labels):
    timestamps = list(raw.keys())
    n_points = len(nfi_labels)
    X = []
    for i in range(n_points):
        row_features = []
        for ts in timestamps:
            row_features.extend(raw[ts][i])
        X.append(row_features)
    X = np.array(X, dtype=float)
    y = nfi_labels["label"].values
    return X,y


def ml(X,y):
    X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )
    # train RF
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    return clf, classification_report(y_test, y_pred)

def main(path, cube):
    labels = load_labels(path)
    labels, geojson = process_labels(labels)
    features = load_features(cube, geojson)
    raw = downlaod_features(features)
    X,y = label_feature_matching(raw, labels)
    clf, report = ml(X,y)
    return clf, report

In [72]:
nfi_labels = geopandas.read_file("nfi_labels.gpkg", layer="nfi")
# reproject to match the cube's CRS 
nfi_labels = nfi_labels.to_crs("EPSG:4326")  

nfi_labels.head()

,label,geometry
0,Eucalyptus,MULTIPOINT ((-8.20735 40.95828))
1,Eucalyptus,MULTIPOINT ((-8.20141 40.96279))
2,Eucalyptus,MULTIPOINT ((-8.19546 40.95379))
3,Eucalyptus,MULTIPOINT ((-8.19547 40.95829))
4,Eucalyptus,MULTIPOINT ((-8.18953 40.95829))


In [73]:
# Drop non forest and other forest
nfi_labels = nfi_labels[nfi_labels["label"]!="Other Forest"]
nfi_labels = nfi_labels[nfi_labels["label"]!="Non forest"]
nfi_labels['label'].value_counts()

label
Shrubland         295
Maritime Pines    157
Eucalyptus         92
Name: count, dtype: int64

## Train ML algorithm

In [ ]:
# Define Resolution
res = 20
# Load Satellite image (for now post image)
s2cube = load_and_sample(current_params, time_post, res)

# Run Main
classifier, report = main("nfi_labels.gpkg", s2cube)

In [98]:
print(report)

                precision    recall  f1-score   support

    Eucalyptus       0.46      0.33      0.39        18
Maritime Pines       0.61      0.62      0.62        32
     Shrubland       0.78      0.83      0.80        59

      accuracy                           0.69       109
     macro avg       0.62      0.60      0.60       109
  weighted avg       0.68      0.69      0.68       109

